# Notebook for the end to end interventions processing

In [1]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

The data contains text documents that are annotated for mentions of participants, interventions and outcomes (PIO) in medical research. For each entity type, P, I, or O, there is a slightly different set of documents in the training and test set. Most of the documents are identical, but each type has a few extra documents. So, let's deal with each type separately for now.

To load the text documents, we first make a list of the document IDs for one entity type (P, I or O):

In [2]:
from pathlib import Path

DATA_DIR = Path("./ebm_nlp_2_00")

docs_dir = DATA_DIR / "documents"

def get_doc_ids(split="train", label_type="participants"):
    """ 
    split: 'train' or 'test' 
    """

    if split == "test":
        split = "test/gold"

    train_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type  # assuming that the split is the same for all entity types, we can just look at one of them
        / split
    )
    
    doc_ids = [p.stem.split(".")[0] for p in train_dir.glob("*.AGGREGATED.ann")]
   # print(doc_ids)
    return sorted(doc_ids)

doc_ids_i = get_doc_ids("train", "interventions")
test_doc_ids_i = get_doc_ids("test", "interventions")

print(f"Number of documents in train split for interventions: {len(doc_ids_i)}")
print(f"Number of documents in test split for interventions: {len(test_doc_ids_i)}")

Number of documents in train split for interventions: 4746
Number of documents in test split for interventions: 187


In [3]:
def load_labels_for_doc(doc_id, label_type="interventions", split="train"):
    """
    label_type: 'participants', 'interventions', or 'outcomes'
    split: 'train' or 'test' 
    """
    if split == "test":
        split = "test/gold"

    ann_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split/ f"{doc_id}.AGGREGATED.ann"
    
    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None
    
    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]
    
    return labels

def load_labels(doc_ids, label_type="interventions", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels

interventions_labels = load_labels(doc_ids_i, "interventions", split="train")

print(f"Length of participants_labels: {len(interventions_labels)}")

test_interventions_labels = load_labels(test_doc_ids_i, "interventions", split="test")
print(f"Length of test_participants_labels: {len(test_interventions_labels)}")

sample = 123
print("Document ID:", doc_ids_i[sample])
print(f"Interventions label example for doc {doc_ids_i[sample]}:")
print(interventions_labels[sample])

Length of participants_labels: 4746
Length of test_participants_labels: 187
Document ID: 10674680
Interventions label example for doc 10674680:
['0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', 

In [4]:
def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_documents(doc_ids):
    documents = []
    for doc_id in doc_ids:
        doc = load_document(doc_id)
        documents.append(doc)
    return documents

interventions_tokens = load_documents(doc_ids_i)
test_interventions_tokens = load_documents(test_doc_ids_i)

# inspect a random element
print("Document ID:", doc_ids_i[sample])
print(f"Tokenised document example for doc {doc_ids_i[sample]}:")
print(interventions_tokens[sample])
print(interventions_labels[sample])

Document ID: 10674680
Tokenised document example for doc 10674680:
['Assessment', 'of', 'therapeutic', 'response', 'of', 'Plasmodium', 'falciparum', 'to', 'chloroquine', 'and', 'sulfadoxine-pyrimethamine', 'in', 'an', 'area', 'of', 'low', 'malaria', 'transmission', 'in', 'Colombia', '.', 'Although', 'chloroquine', '(', 'CQ', ')', 'resistance', 'was', 'first', 'reported', 'in', 'Colombia', 'in', '1961', 'and', 'sulfadoxine-pyrimethamine', '(', 'SP', ')', 'resistance', 'in', '1981', ',', 'the', 'frequency', 'of', 'treatment', 'failures', 'to', 'these', 'drugs', 'in', 'Colombia', 'is', 'unclear', '.', 'A', 'modified', 'World', 'Health', 'Organization', '14-day', 'in', 'vivo', 'drug', 'efficacy', 'test', 'for', 'uncomplicated', 'Plasmodium', 'falciparum', 'malaria', 'in', 'areas', 'with', 'intense', 'malaria', 'transmission', 'was', 'adapted', 'to', 'reflect', 'the', 'clinical', 'and', 'epidemiologic', 'features', 'of', 'a', 'low-intensity', 'malaria', 'transmission', 'area', 'in', 'the', 

### Process to follow:
1. Load the files
2. Convert labels to ones combining the B I O labels with the ner tags
3. Convert combined labels to numeric ids
4. Initialize the model and tokenizer
5. Tokenize the words and align the labels with any words that have been split into multiple tokens
6. Prepare the model
7. Create the compute metrics function
8. Add the training arguments
9. Run the model
10. Repeat for Participants and Outcomes

In [5]:
from itertools import chain
import numpy as np

all_labels = chain(*interventions_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['0' '1' '2' '3' '4' '5' '6' '7']


In [6]:
def hierarchical_to_ner(tags):
    """
    Convert EBM-NLP hierarchical labels (0–4) to flat BIO tags.

    Parameters
    ----------
    tags : list[int]
        A list of hierarchical labels for a single document.

    Returns
    -------
    list[str]
        BIO tags ("O", "B", "I").
    """

    bio = []
    prev = 0

    for t in tags:
        t = int(t)  # ensure it's an integer
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B-" + str(t))
            else:
                bio.append("I-" + str(t))
        prev = t
        

    return bio

def convert_all_labels_to_ner(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_ner(doc_labels)
    return labels

interventions_ner_labels = convert_all_labels_to_ner(interventions_labels)
test_interventions_ner_labels = convert_all_labels_to_ner(test_interventions_labels)

all_labels = chain(*interventions_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['B-1' 'B-2' 'B-3' 'B-4' 'B-5' 'B-6' 'B-7' 'I-1' 'I-2' 'I-3' 'I-4' 'I-5'
 'I-6' 'I-7' 'O']


Next define the mappings from id to label and label to id and use this to create the labels

In [7]:
id2label_i = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
    9: "B-5",
    10: "I-5",
    11: "B-6",
    12: "I-6",
    13: "B-7",
    14: "I-7",
}

label2id_i = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
    "B-5": 9,
    "I-5": 10,
    "B-6": 11,
    "I-6": 12,
    "B-7": 13,
    "I-7": 14,
}

Then we convert the ner tags to their respective ids

In [8]:
def label_to_ids(labels_list, labels2ids):
    ids_list = []
    for label_list in labels_list:
        ids = []
        for label in label_list:
            ids.append(labels2ids[label])
        ids_list.append(ids)
    return ids_list

In [9]:
interventions_ner_ids = label_to_ids(interventions_ner_labels, label2id_i)
test_interventions_ner_ids = label_to_ids(test_interventions_ner_labels, label2id_i)
print(interventions_ner_ids[0])
print(test_interventions_ner_ids[0])

[0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 5, 6, 6, 6, 0, 5, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Now since we are using a pretrained model we can use the pretrained tokenization to tokenize our input words. However we need to be careful when doing this as some words may be broken up into several tokens resulting in the tokens and labels being misaligned. To solve this issue we can use the Huggingface inbuilt tokenizer field `batch_id` which keeps track of multiple tokens belonging to the same word

In [10]:
def tokenize_and_align_labels(inputs):
    tokenized_inputs = tokenizer(inputs["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(inputs["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i) # This gives the same word id for words that have been split up
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx: # Only label the first token of a given word
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [11]:
from datasets import Dataset, DatasetDict

interventions_initial_train_ds = Dataset.from_dict({
    "tokens": interventions_tokens,
    "ner_tags": interventions_ner_ids,
})

test_interventions_initial_train_ds = Dataset.from_dict({
    "tokens": test_interventions_tokens,
    "ner_tags": test_interventions_ner_ids,
})

In [43]:
ds_dict_i = DatasetDict({
    "train": interventions_initial_train_ds,
    "test": test_interventions_initial_train_ds,
})

Now to use the above function we first need to initialize our tokenizer. Our first test will be the smallest BERT model DistilBert (base-uncased)

In [13]:
from transformers import AutoTokenizer, DistilBertForTokenClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [44]:
interventions_ds = ds_dict_i.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4746 [00:00<?, ? examples/s]

Map:   0%|          | 0/187 [00:00<?, ? examples/s]

In [15]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [16]:
import evaluate

seqeval = evaluate.load("seqeval")

In [39]:
import numpy as np

label_list_i = list(label2id_i.keys())

def compute_metrics_i(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [40]:
model_i = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 15, id2label=id2label_i, label2id=label2id_i)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [41]:
from transformers import TrainingArguments, Trainer

training_args_i = TrainingArguments(
    output_dir = "interventions_classification_model",
    learning_rate = 2e-4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 2,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

trainer_i = Trainer(
    model = model_i,
    args = training_args_i,
    train_dataset = interventions_ds["train"],
    eval_dataset = interventions_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_i,
)

In [42]:
trainer_i.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.164567,0.481584,0.354817,0.408594,0.954783
2,0.211400,0.149714,0.462049,0.425373,0.442953,0.951044


/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/billy/miniconda3/envs/text_analytics/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=594, training_loss=0.20152013309876926, metrics={'train_runtime': 107.3337, 'train_samples_per_second': 88.435, 'train_steps_per_second': 5.534, 'total_flos': 1235068703291124.0, 'train_loss': 0.20152013309876926, 'epoch': 2.0})

## Participants
Now to repeat the process with participants

In [55]:
doc_ids_p = get_doc_ids("train", "participants")
test_doc_ids_p = get_doc_ids("test", "participants")

print(f"Number of documents in train split for participants: {len(doc_ids_p)}")
print(f"Number of documents in test split for participants: {len(test_doc_ids_p)}")

participants_tokens = load_documents(doc_ids_p)
test_participants_tokens = load_documents(test_doc_ids_p)

Number of documents in train split for participants: 4609
Number of documents in test split for participants: 189


In [25]:
participants_labels = load_labels(doc_ids_p, "participants", split="train")
print(f"Length of participants_labels: {len(participants_labels)}")

test_participants_labels = load_labels(test_doc_ids_p, "participants", split="test")
print(f"Length of test_participants_labels: {len(test_participants_labels)}")

participants_ner_labels = convert_all_labels_to_ner(participants_labels)
test_participants_ner_labels = convert_all_labels_to_ner(test_participants_labels)

Length of participants_labels: 4609
Length of test_participants_labels: 189


In [26]:
all_labels_p = chain(*participants_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels_p)))

['B-1' 'B-2' 'B-3' 'B-4' 'I-1' 'I-2' 'I-3' 'I-4' 'O']


This gives us the labels for the participants

In [28]:
id2label_p = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
}

label2id_p = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
}

In [63]:
participants_ner_ids = label_to_ids(participants_ner_labels, label2id_p)
test_participants_ner_ids = label_to_ids(test_participants_ner_labels, label2id_p)

In [31]:
participants_initial_train_ds = Dataset.from_dict({
    "tokens": participants_tokens,
    "ner_tags": participants_ner_ids,
})

test_participants_initial_train_ds = Dataset.from_dict({
    "tokens": test_participants_tokens,
    "ner_tags": test_participants_ner_ids,
})

In [45]:
ds_dict_p = DatasetDict({
    "train": participants_initial_train_ds,
    "test": test_participants_initial_train_ds,
})

In [48]:
participants_ds = ds_dict_p.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4609 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

In [46]:
label_list_p = list(label2id_p.keys())

def compute_metrics_p(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_p[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list_p[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [47]:
model_p = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 9, id2label=id2label_p, label2id=label2id_p)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [64]:
training_args_p = TrainingArguments(
    output_dir = "participants_classification_model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

trainer_p = Trainer(
    model = model_p,
    args = training_args_p,
    train_dataset = participants_ds["train"],
    eval_dataset = participants_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_p,
)

In [65]:
trainer_p.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.194341,0.336028,0.341149,0.338569,0.948983
2,0.054900,0.178090,0.323372,0.355217,0.338547,0.948722
3,0.054900,0.189002,0.333697,0.358734,0.345763,0.949044


TrainOutput(global_step=867, training_loss=0.057248860219495656, metrics={'train_runtime': 158.6947, 'train_samples_per_second': 87.13, 'train_steps_per_second': 5.463, 'total_flos': 1797318230782662.0, 'train_loss': 0.057248860219495656, 'epoch': 3.0})

## Outcomes

In [56]:
doc_ids_o = get_doc_ids("train", "outcomes")
test_doc_ids_o = get_doc_ids("test", "outcomes")

print(f"Number of documents in train split for outcomes: {len(doc_ids_o)}")
print(f"Number of documents in test split for outcomes: {len(test_doc_ids_o)}")

outcomes_tokens = load_documents(doc_ids_o)
test_outcomes_tokens = load_documents(test_doc_ids_o)

Number of documents in train split for outcomes: 4681
Number of documents in test split for outcomes: 190


In [58]:
outcomes_labels = load_labels(doc_ids_o, "outcomes", split="train")
print(f"Length of outcomes_labels: {len(outcomes_labels)}")

test_outcomes_labels = load_labels(test_doc_ids_o, "outcomes", split="test")
print(f"Length of test_outcomes_labels: {len(test_outcomes_labels)}")

outcomes_ner_labels = convert_all_labels_to_ner(outcomes_labels)
test_outcomes_ner_labels = convert_all_labels_to_ner(test_outcomes_labels)

Length of outcomes_labels: 4681
Length of test_outcomes_labels: 190


In [59]:
all_labels_o = chain(*outcomes_ner_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels_o)))

['B-1' 'B-2' 'B-3' 'B-4' 'B-5' 'B-6' 'I-1' 'I-2' 'I-3' 'I-4' 'I-5' 'I-6'
 'O']


In [61]:
id2label_o = {
    0: "O",
    1: "B-1",
    2: "I-1",
    3: "B-2",
    4: "I-2",
    5: "B-3",
    6: "I-3",
    7: "B-4",
    8: "I-4",
    9: "B-5",
    10: "I-5",
    11: "B-6",
    12: "I-6",
}

label2id_o = {
    "O": 0,
    "B-1": 1,
    "I-1": 2,
    "B-2": 3,
    "I-2": 4,
    "B-3": 5,
    "I-3": 6,
    "B-4": 7,
    "I-4": 8,
    "B-5": 9,
    "I-5": 10,
    "B-6": 11,
    "I-6": 12,
}

In [62]:
outcomes_ner_ids = label_to_ids(outcomes_ner_labels, label2id_o)
test_outcomes_ner_ids = label_to_ids(test_outcomes_ner_labels, label2id_o)

In [66]:
outcomes_initial_train_ds = Dataset.from_dict({
    "tokens": outcomes_tokens,
    "ner_tags": outcomes_ner_ids,
})

test_outcomes_initial_train_ds = Dataset.from_dict({
    "tokens": test_outcomes_tokens,
    "ner_tags": test_outcomes_ner_ids,
})

In [67]:
ds_dict_o = DatasetDict({
    "train": outcomes_initial_train_ds,
    "test": test_outcomes_initial_train_ds,
})

In [69]:
outcomes_ds = ds_dict_o.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/4681 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

In [70]:
label_list_o = list(label2id_o.keys())

def compute_metrics_o(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list_o[p] for (p, l) in zip(prediction, label) if l != -100] # Use this to ignore everything but the first token for words that got split during tokenization
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list_o[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [71]:
model_o = DistilBertForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels = 13, id2label=id2label_o, label2id=label2id_o)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [74]:
training_args_o = TrainingArguments(
    output_dir = "outcomes_classification_model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 10,
    weight_decay = 0.01,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    push_to_hub = False,
)

trainer_o = Trainer(
    model = model_o,
    args = training_args_o,
    train_dataset = outcomes_ds["train"],
    eval_dataset = outcomes_ds["test"],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics_o,
)

In [75]:
trainer_o.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.195594,0.268844,0.305061,0.285810,0.930403
2,0.209900,0.197679,0.284381,0.325731,0.303654,0.929984
3,0.209900,0.194740,0.290545,0.300071,0.295231,0.932778
4,0.186300,0.192425,0.297243,0.315039,0.305882,0.932000
5,0.186300,0.206646,0.293033,0.305773,0.299268,0.929306
6,0.161500,0.210069,0.273981,0.325731,0.297623,0.927769
7,0.144500,0.215640,0.285132,0.299359,0.292072,0.929585
8,0.144500,0.213379,0.292568,0.308624,0.300382,0.930064
9,0.130200,0.221453,0.272280,0.315752,0.292409,0.927689
10,0.130200,0.220702,0.279245,0.316465,0.296692,0.928607


TrainOutput(global_step=2930, training_loss=0.16032560953914915, metrics={'train_runtime': 541.0244, 'train_samples_per_second': 86.521, 'train_steps_per_second': 5.416, 'total_flos': 6084369753692700.0, 'train_loss': 0.16032560953914915, 'epoch': 10.0})

In [ ]:
trainer_o.